# Regional GAA Profiling
Owner: Anje
Table: `pondolens.dev_bronze.regional_expenditure_gaa`
Baseline: 4,780 rows

**Goal:** identify population records suitable for Q3 and required Silver treatments. All queries are read-only.

**Row meaning:** A government expenditure allocation for a specific fiscal year, expenditure category, particular (department or special purpose fund), and geographical area.

## Checks
- Check counts by file/year, column types, and nulls/blanks.
- Define what one row represents; check exact duplicates and repeated business keys.
- Verify regions, years, categories, particulars, numeric amounts, units, and price basis.
- Identify totals/subtotals to avoid double counting.
- Compare sample amounts and totals with the original source.
- With Joy, check whether GAA and obligations have matching scope, periods, and regions.
- Check regional compatibility with population and GRDP.

## Findings
| Check / Issue | Affected rows / Evidence | Proposed Silver treatment |
|---|---|---|
| Dashes in expenditure_amount column| 336 rows in 2025 table, 344 rows in 2024 table| treat as ZERO|
| Special characters in particulars column (). The data contains both the department and the acronym of the department inside the () | 40 distinct column_value with 20 occurences each for both 2024 and 2025 tables. 39 of these values are the agency names while the other value is GAA(updated)| Separate the acronym from the full agency names|
| when particulars = 'All', these are aggregate rows. Other particulars are department or fund detail rows rathen than automatically being subtotals | All rows| Keep all rows in Silver and add two classification columns. This preserves published totals for validation while allowing dashboards to select detail rows without double counting. financial_row_type: grand_total, category_total, detail AND geographic_row_type: geographic_total, nonregional, regional|
|Expenditure amount in thousand pesos|all rows| convert expenditure amount to be in thousand pesos and remove unit_amoount column|

## Assumptions to Verify
| Question | Status |
|---|---|
|Dashes have a value of zero|to verify with team|
|Should we convert all Region-IVA to RegionIVA|to verify with team|

## Semantic Notes
- The amount is expressed in thousand Philippine pesos (₱'000).
- 'All' represents category subtotals
- 'Total' represents the sum of applicable geographical allocations


## Conclusion
- Required Silver changes: Treat dashes as zero, separate the full agency names from acronyms, convert expenditure amount to be in thousand pesos, convert region-IVA to region IVA
- Ready for Q1/Q3? Why / why not: Ready for Q1 and Q3. 
- Open questions: How do we treat Nationawide and Central Office allocations?

## 1. Structure and coverage
Inspect columns, sample records, years, and geographic labels. PHILIPPINES is a national total, not a region.

In [0]:
%sql
DESCRIBE TABLE pondolens.dev_bronze.regional_expenditure_gaa;

In [0]:
%sql
SELECT *
FROM pondolens.dev_bronze.regional_expenditure_gaa
LIMIT 10;

In [0]:
%sql
-- List of all regions

SELECT
    DISTINCT(regions) as region_list
FROM pondolens.dev_bronze.regional_expenditure_gaa

In [0]:
%sql
SELECT COUNT(*)
FROM pondolens.dev_bronze.regional_expenditure_gaa;

## 2. Data quality
Check required values, numeric/date parsing, rescued data, and repeated candidate keys. Empty duplicate results mean no repeated keys were found.


In [0]:
%sql
-- Null counts
SELECT
    COUNT(*) as total_rows,
    COUNT_IF(source_name IS NULL) as source_name_nulls,
    COUNT_IF(source_page IS NULL) as source_page_nulls,
    COUNT_IF(price_basis IS NULL) as price_basis_nulls,
    COUNT_IF(year IS NULL) as year_nulls,
    COUNT_IF(category IS NULL) as category_nulls,
    COUNT_IF(particulars IS NULL) as particulars_nulls,
    COUNT_IF(regions IS NULL) as regions_nulls,
    COUNT_IF(expenditure_amount IS NULL) as expenditure_amount_nulls,
    COUNT_IF(unit_amount IS NULL) as unit_amount_nulls
FROM
    pondolens.dev_bronze.regional_expenditure_gaa;

In [0]:
%sql
-- Duplicates
SELECT
    source_name,
    source_page,
    price_basis,
    year,
    category,
    particulars,
    regions,
    expenditure_amount,
    unit_amount,
    COUNT(*) AS duplicate_count
FROM pondolens.dev_bronze.regional_expenditure_gaa
GROUP BY
    source_name,
    source_page,
    price_basis,
    year,
    category,
    particulars,
    regions,
    expenditure_amount,
    unit_amount
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Check for special characters 

WITH column_values AS (
    SELECT
        source_name AS file_name,
        STACK(
            9,
            'source_name', CAST(source_name AS STRING),
            'source_page', CAST(source_page AS STRING),
            'price_basis', CAST(price_basis AS STRING),
            'year', CAST(year AS STRING),
            'category', CAST(category AS STRING),
            'particulars', CAST(particulars AS STRING),
            'regions', CAST(regions AS STRING),
            'expenditure_amount', CAST(expenditure_amount AS STRING),
            'unit_amount', CAST(unit_amount AS STRING)
        ) AS (column_name, column_value)
    FROM pondolens.dev_bronze.regional_expenditure_gaa
)
SELECT
    file_name,
    column_name,
    column_value,
    COUNT(*) AS occurrences
FROM column_values
WHERE column_value RLIKE '[^a-zA-Z0-9 ]'
GROUP BY file_name, column_name, column_value
ORDER BY column_name, occurrences DESC;

## 3. Reconcile totals
- Geographical reconciliation: Does the sum of all regional amounts, Nationwide, and Central Office equal the reported Total?
- Particulars reconciliation: Does the sum of individual departments or special purpose funds equal the corresponding All subtotal?
- Grand total reconciliation: Does DEPARTMENTS + SPECIAL PURPOSE FUNDS equal GRAND TOTAL?

In [0]:
%sql
WITH reconciliation AS (
    SELECT
        year,
        category,
        particulars,

        SUM(
            CASE WHEN regions = 'Total'
            THEN TRY_CAST(expenditure_amount AS DECIMAL(20,2))
            ELSE 0 END
        ) AS reported_total,

        SUM(
            CASE WHEN regions <> 'Total'
            THEN TRY_CAST(expenditure_amount AS DECIMAL(20,2))
            ELSE 0 END
        ) AS calculated_total

    FROM pondolens.dev_bronze.regional_expenditure_gaa
    GROUP BY year, category, particulars
)

SELECT
    *,
    reported_total - calculated_total AS difference,
    CASE
        WHEN reported_total = calculated_total THEN 'MATCH'
        ELSE 'ISSUE'
    END AS status
FROM reconciliation
ORDER BY category, particulars;

In [0]:
%sql
WITH reconciliation AS (
    SELECT
        year,
        category,
        regions,

        SUM(
            CASE WHEN particulars = 'All'
            THEN TRY_CAST(expenditure_amount AS DECIMAL(20,2))
            ELSE 0 END
        ) AS reported_total,

        SUM(
            CASE WHEN particulars <> 'All'
            THEN TRY_CAST(expenditure_amount AS DECIMAL(20,2))
            ELSE 0 END
        ) AS calculated_total

    FROM pondolens.dev_bronze.regional_expenditure_gaa
    WHERE category IN (
        'DEPARTMENTS',
        'SPECIAL PURPOSE FUNDS'
    )
    GROUP BY year, category, regions
)

SELECT
    *,
    reported_total - calculated_total AS difference,
    CASE
        WHEN reported_total = calculated_total THEN 'MATCH'
        ELSE 'ISSUE'
    END AS status
FROM reconciliation
ORDER BY category, regions;

In [0]:
%sql
WITH reconciliation AS (
    SELECT
        year,
        regions,

        SUM(
            CASE
                WHEN category = 'GRAND TOTAL'
                AND particulars = 'All'
                THEN TRY_CAST(expenditure_amount AS DECIMAL(20,2))
                ELSE 0
            END
        ) AS reported_grand_total,

        SUM(
            CASE
                WHEN category IN (
                    'DEPARTMENTS',
                    'SPECIAL PURPOSE FUNDS'
                )
                AND particulars = 'All'
                THEN TRY_CAST(expenditure_amount AS DECIMAL(20,2))
                ELSE 0
            END
        ) AS calculated_grand_total

    FROM pondolens.dev_bronze.regional_expenditure_gaa
    GROUP BY year, regions
)

SELECT
    *,
    reported_grand_total - calculated_grand_total AS difference,
    CASE
        WHEN reported_grand_total = calculated_grand_total
        THEN 'MATCH'
        ELSE 'ISSUE'
    END AS status
FROM reconciliation
ORDER BY regions;